# How large language models work: worked solution

Worked solution for the exercise in the lesson **[How large language models work](https://ramkumarmd.com/medical-ai/generative-ai-llms/how-llms-work)** (*Generative AI & LLMs* track, Medical AI course by Dr. Ram Kumar G). Try the [exercise notebook](01-how-llms-work.ipynb) first. Other answers can be just as good: the written parts show one reasonable answer, not the only one.

**In short:** A large language model (LLM) generates text one token at a time by predicting what is most likely to come next, based on patterns learned from enormous amounts of text. Further training on instructions and human feedback turns this into a helpful assistant. Because the model optimises for plausible text rather than truth, it can be fluent, knowledgeable and still confidently wrong, which is why clinical use needs grounding, checking and human responsibility.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Predicting the next token

Invented scores for the next word, and the probabilities at three temperatures.

In [ ]:
import numpy as np

# A made-up model's scores (logits) for the next word after:
# "Metformin is usually first-line treatment for type 2 ..."
candidates = ["diabetes", "patients", "adults", "disease", "hypertension"]
logits = np.array([6.0, 3.5, 2.8, 2.0, 0.5])

def softmax(z, temperature=1.0):
    z = z / temperature
    e = np.exp(z - z.max())
    return e / e.sum()

for t in [0.2, 1.0, 2.0]:
    probs = softmax(logits, t)
    print(f"temperature {t}: " + ", ".join(f"{w} {p:.2f}" for w, p in zip(candidates, probs)))

# temperature 0.2: diabetes 1.00, patients 0.00, adults 0.00, disease 0.00, hypertension 0.00
# temperature 1.0: diabetes 0.87, patients 0.07, adults 0.04, disease 0.02, hypertension 0.00
# temperature 2.0: diabetes 0.59, patients 0.17, adults 0.12, disease 0.08, hypertension 0.04

### Why fluent is not the same as true

A bigram model: the simplest possible language model, trained on five sentences.

In [ ]:
import numpy as np
from collections import Counter, defaultdict

corpus = """the patient was started on metformin .
the patient was started on insulin .
the patient was referred to cardiology .
the trial was started in 2019 .
the trial was referred to the ethics committee .""".split("\n")

following = defaultdict(Counter)
for sentence in corpus:
    words = sentence.split()
    for a, b in zip(words, words[1:]):
        following[a][b] += 1

rng = np.random.default_rng(4)
def generate(start="the", max_words=12):
    words = [start]
    while words[-1] != "." and len(words) < max_words:
        options = following[words[-1]]
        probs = np.array(list(options.values())) / sum(options.values())
        words.append(rng.choice(list(options), p=probs))
    return " ".join(words)

for _ in range(5):
    print(generate())

# the ethics committee .
# the patient was started in 2019 .
# the trial was started on insulin .
# the patient was referred to the trial was referred to cardiology .
# the trial was started on insulin .

## Exercise

**Goal:** Explore how next-token sampling and temperature create variability and plausible errors, using toy models that run without any API.

**Data:** Synthetic sentences written in the notebook (no real patient data) (Generated in the notebook)

### Step 1

Reproduce the temperature table and add a 'top-k' sampling rule that only allows the 3 most likely tokens.

In [ ]:
def top_k_softmax(z, k=3, temperature=1.0):
    """Keep only the k highest-scoring tokens, then apply temperature and renormalise."""
    keep = np.argsort(z)[-k:]
    masked = np.full_like(z, -np.inf)
    masked[keep] = z[keep]
    return softmax(masked, temperature)

for t in [0.2, 1.0, 2.0]:
    plain, top3 = softmax(logits, t), top_k_softmax(logits, 3, t)
    print(f"temperature {t}:")
    print("  all tokens: " + ", ".join(f"{w} {p:.2f}" for w, p in zip(candidates, plain)))
    print("  top-3:      " + ", ".join(f"{w} {p:.2f}" for w, p in zip(candidates, top3)))

Top-k sampling removes the long tail: however high the temperature, "disease" and "hypertension" can no
longer be chosen. It limits how strange an output can get, but it does not make the output true; the
remaining tokens can still be wrong in context.

### Step 2

Extend the bigram corpus with ten more synthetic sentences and count how many generated sentences are true statements about the corpus.

In [ ]:
extra = """the patient was started on empagliflozin .
the patient was referred to ophthalmology .
the patient was reviewed in clinic .
the trial was registered in 2018 .
the trial was stopped early .
the nurse was trained in 2020 .
the nurse was referred to occupational health .
the dose was reduced to 500 mg .
the dose was increased in clinic .
the clinic was started in 2015 .""".split("\n")
full_corpus = corpus + extra

def build_model(sentences, context=1):
    model = defaultdict(Counter)
    for sentence in sentences:
        words = ["<s>"] * context + sentence.split()
        for i in range(context, len(words)):
            model[tuple(words[i - context:i])][words[i]] += 1
    return model

def sample(model, context=1, seed=0, n=500, max_words=15):
    r = np.random.default_rng(seed)
    out = []
    for _ in range(n):
        words = ["<s>"] * context
        while words[-1] != "." and len(words) < max_words + context:
            options = model[tuple(words[-context:])]
            probs = np.array(list(options.values())) / sum(options.values())
            words.append(r.choice(list(options), p=probs))
        out.append(" ".join(words[context:]))
    return out

known = set(full_corpus)
bigram = build_model(full_corpus, context=1)
generated = sample(bigram, context=1)
true_share = np.mean([s in known for s in generated])
print(f"Bigram model: {true_share:.0%} of {len(generated)} generated sentences appear in the corpus")
print("Examples of new (false) sentences:")
for s in [s for s in dict.fromkeys(generated) if s not in known][:5]:
    print("  ", s)

"Appears in the corpus" is our stand-in for "true": every sentence in the corpus is a fact about this toy
world, and a new combination ("the nurse was started on empagliflozin") is a fluent statement that nobody made.
Each word follows plausibly from the one before it, which is all the model checks.

### Step 3

Change the bigram model to a trigram model (two words of context) and compare how often it produces false combinations.

In [ ]:
trigram = build_model(full_corpus, context=2)
generated3 = sample(trigram, context=2)
print(f"Bigram:  {1 - np.mean([s in known for s in generated]):.0%} of generated sentences are not in the corpus")
print(f"Trigram: {1 - np.mean([s in known for s in generated3]):.0%} of generated sentences are not in the corpus")

With two words of context the model can tell "patient was" from "trial was", so far fewer false
combinations appear. On a corpus this small, a trigram model mostly copies its training sentences. Real
LLMs use thousands of tokens of context and are trained on vast text, so they rarely produce word salad,
but the same principle holds: they generate what is *likely* given the context, and a likely sentence can
still be false, especially for rare facts the training data covered thinly.

### Step 4

Optional, with an approved model: ask the same factual question five times at two temperatures and record how the answers vary.

Optional, and only with a model your organisation has approved, using a question with no patient data.
The experiment needs one function that sends a prompt and returns text; every provider's SDK offers an
equivalent, so the harness below leaves `ask` for you to fill in (API keys belong in environment variables).

```python
import os
from collections import Counter

def ask(prompt: str, temperature: float) -> str:
    """Call your approved model here and return its text answer."""
    raise NotImplementedError

QUESTION = "In one sentence: what is the usual first-line glucose-lowering medicine for type 2 diabetes in adults?"
for temperature in [0.0, 1.0]:
    answers = [ask(QUESTION, temperature) for _ in range(5)]
    print(f"temperature {temperature}: {len(set(answers))} different answers")
    for answer, n in Counter(answers).most_common():
        print(f"  {n} x {answer}")
```

What to record: how many distinct answers you get at each temperature, whether the *facts* change or only
the wording, and whether any answer adds a claim that is wrong. Even at temperature 0, some services do not
return identical text every time, so a single run is never enough to judge a model.

---

Lesson: [https://ramkumarmd.com/medical-ai/generative-ai-llms/how-llms-work](https://ramkumarmd.com/medical-ai/generative-ai-llms/how-llms-work). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.